## Cell 0 — Runtime probe and installation

In [ ]:
# =============================================================
# CELL 0 — Runtime probe + dependencies
# =============================================================
import os, sys, subprocess, platform, textwrap

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

# ---- 0.1 Probe the machine we were actually given -------------
import multiprocessing
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU cores   :", multiprocessing.cpu_count(), "  <-- drives ORC runtime (see PRD 5.1)")
try:
    import psutil
    print("System RAM  : %.1f GB" % (psutil.virtual_memory().total / 1e9))
except ImportError:
    sh("cat /proc/meminfo | head -1")

try:
    import torch
    print("Torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("GPU         :", p.name, "| VRAM: %.1f GB" % (p.total_memory / 1e9))
except ImportError:
    print("Torch not present yet.")

# ---- 0.2 Install ---------------------------------------------
sh("pip install -q GraphRicciCurvature POT networkx>=3.4 pyarrow tqdm")
sh("pip install -q torch-geometric")
sh("pip install -q transformers sentence-transformers adapters")

# ---- 0.3 Drive for checkpoints -------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/orca_gcn_caif'
except Exception:
    CACHE_ROOT = './orca_gcn_caif'          # local / CI fallback
os.makedirs(CACHE_ROOT, exist_ok=True)
print("Cache root  :", CACHE_ROOT)


## Cell 1 — Master configuration

In [ ]:
# =============================================================
# CELL 1 — CFG: the single source of every tunable value
# =============================================================
from dataclasses import dataclass, asdict, field
from typing import Optional, Tuple
import json, hashlib, os

@dataclass
class Config:
    # ---------- Phase 1: acquisition ----------
    OPENALEX_API_KEY: str = ""             # REQUIRED since 2026-02-13
    OPENALEX_BASE: str = "https://api.openalex.org"
    JOURNAL_IDS: Tuple[str, ...] = ("S137773608",)   # OpenAlex source IDs
    YEAR_START: int = 2022
    YEAR_END: int = 2023
    MAX_SEED_PAPERS: int = 20000
    MAX_CITERS: Optional[int] = None       # None => unbounded (PRD default).
                                            # citing_works() has no cap of its
                                            # own otherwise -- set this for
                                            # dev/quota safety (found via live
                                            # testing: 25 seeds pulled 26,600+
                                            # citer records and was still
                                            # climbing when interrupted).
    CITES_BATCH: int = 50                  # OR-filter IDs per request
    PER_PAGE: int = 200
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 5
    SELECT_FIELDS: str = ("id,doi,title,publication_year,publication_date,"
                          "primary_location,authorships,referenced_works,"
                          "topics,abstract_inverted_index,cited_by_count")

    # ---------- Phase 2a: ORC ----------
    ORC_ALPHA: float = 0.5                 # random-walk laziness
    ORC_METHOD: str = "OTDSinkhornMix"     # OTD | Sinkhorn | OTDSinkhornMix
    ORC_PROC_CAP: int = 12
    ORC_ISOLATED_VALUE: float = 0.0

    # ---------- Phase 2b: ORCA-GCN ----------
    TAU_CURV: float = 1.0
    GATE_MODE: str = "softplus"            # softplus | exp | sigmoid
    GCN_LAYERS: int = 2
    GCN_HIDDEN: int = 128
    GCN_DROPOUT: float = 0.5
    GCN_LR: float = 0.01
    GCN_EPOCHS: int = 200
    GCN_USE_TEXT_FEATURES: bool = True
    GCN_TEXT_PCA_DIM: int = 64

    # ---------- Phase 2c: temporal ----------
    LAMBDA_DECAY: Optional[float] = None   # None => MLE-fit per field (PRD 4.3)
    TEMPORAL_IMPOSSIBLE_VALUE: float = 0.0

    # ---------- Phase 2d: semantic ----------
    SEMANTIC_BACKEND: str = "specter2"     # specter2 | st-specter | scibert
    EMB_BATCH_SIZE: int = 32
    EMB_MAX_TOKENS: int = 512
    EMB_FP16: bool = True
    SEMANTIC_MISSING_VALUE: float = 0.5

    # ---------- Phase 2e: fusion ----------
    STANDARDIZE: str = "robust"            # robust | zscore | none
    FUSION_ORDER: Tuple[str, ...] = ("kappa", "h", "tau", "sigma")

    # ---------- Phase 3: trust head ----------
    LOGIT_CLAMP: float = 30.0              # PRD 2.3.3 — Test 11 depends on this
    HEAD_LR: float = 0.01
    HEAD_L2: float = 1e-4
    HEAD_EPOCHS: int = 300
    HEAD_POS_WEIGHT: Optional[float] = None   # None => auto from class balance
    WEAK_CARTEL_PCT: float = 0.99
    WEAK_AUTHOR_OVERLAP: float = 0.50
    WEAK_BULK_REF_MIN: int = 5
    WEAK_POS_MIN_CITED: int = 10

    # ---------- Phase 4: metric + validation ----------
    FIELD_PEER_K: int = 20
    NULL_MODE: str = "strict"              # fast | strict
    NULL_B_FAST: int = 500
    NULL_B_STRICT: int = 50
    NULL_SWAP_FACTOR: int = 10
    Z_THRESHOLD: float = 1.96
    RECAL_VAR_TRIGGER: float = 0.30
    RECAL_MAX_PASSES: int = 2

    # ---------- infrastructure ----------
    SEED: int = 42
    USE_CACHE: bool = True
    CACHE_ROOT: str = CACHE_ROOT
    LOG_LEVEL: str = "INFO"

CFG = Config()

def _resolve_api_key():
    """Colab Secrets first (never paste a key into a cell or chat), then
    the OPENALEX_API_KEY env var for local/CI runs. See README.md."""
    try:
        from google.colab import userdata
        key = userdata.get("OPENALEX_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENALEX_API_KEY", "")

CFG.OPENALEX_API_KEY = _resolve_api_key()
assert CFG.OPENALEX_API_KEY, (
    "No OpenAlex API key found. In Colab: Secrets panel (key icon, left "
    "sidebar) -> add OPENALEX_API_KEY -> enable notebook access. "
    "Locally: set the OPENALEX_API_KEY env var. Get a free key at "
    "openalex.org/settings/api (PRD 3.5)."
)

def cfg_hash(c) -> str:
    return hashlib.sha256(json.dumps(asdict(c), sort_keys=True,
                                     default=str).encode()).hexdigest()[:12]

CFG_HASH = cfg_hash(CFG)
print("Config hash:", CFG_HASH)
with open(os.path.join(CFG.CACHE_ROOT, f"cfg_{CFG_HASH}.json"), "w") as f:
    json.dump(asdict(CFG), f, indent=2, default=str)


## Cell 2 — Utilities: cache, retry, determinism

In [ ]:
# =============================================================
# CELL 2 — Infrastructure utilities
# =============================================================
import os, sys, gzip, pickle, random, logging, time, functools
import numpy as np, pandas as pd

# stream=sys.stdout (not the StreamHandler default of sys.stderr) + force=True:
# ipykernel swaps sys.stdout/stderr for its own per-cell-output stream, but a
# StreamHandler binds whatever `sys.stderr` object exists AT CONSTRUCTION TIME.
# If logging is configured before that swap settles (or basicConfig() no-ops on
# a rerun because handlers already exist), log.info() calls silently vanish
# instead of reaching the visible cell output -- print() still works because
# it re-reads sys.stdout on every call rather than caching a reference.
logging.basicConfig(level=getattr(logging, CFG.LOG_LEVEL),
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("caif")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass
set_seed(CFG.SEED)

# ---- checkpoint helpers: every expensive cell uses these ------
# pickle is used only for this notebook's own Drive cache (CFG.CACHE_ROOT):
# files this same run wrote and reads back, never an externally-sourced
# artifact. Not a deserialization-of-untrusted-input risk in this context.
def _p(name, ext): return os.path.join(CFG.CACHE_ROOT, f"{name}.{ext}")

def cache_exists(name, ext="pkl"): return CFG.USE_CACHE and os.path.exists(_p(name, ext))

def cache_save(name, obj, ext="pkl"):
    with open(_p(name, ext), "wb") as f: pickle.dump(obj, f, protocol=4)
    log.info("cached -> %s", _p(name, ext))

def cache_load(name, ext="pkl"):
    with open(_p(name, ext), "rb") as f: obj = pickle.load(f)
    log.info("loaded <- %s", _p(name, ext)); return obj

def stage(name, ext="pkl"):
    """Decorator: skip an expensive stage entirely if its artifact exists."""
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **kw):
            if cache_exists(name, ext): return cache_load(name, ext)
            t0 = time.time(); out = fn(*a, **kw)
            log.info("stage %-12s took %.1fs", name, time.time() - t0)
            cache_save(name, out, ext); return out
        return wrap
    return deco

def safe_finite(x, nan=0.0, posinf=1.0, neginf=-1.0):
    """Test 03: no NaN / Inf may ever enter the feature vector."""
    return np.nan_to_num(np.asarray(x, dtype=np.float64),
                         nan=nan, posinf=posinf, neginf=neginf)


## Cell 3 — Phase 1: OpenAlex acquisition

In [ ]:
# =============================================================
# CELL 3 — PHASE 1a: OpenAlex acquisition (quota-aware)
# =============================================================
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

class OpenAlexClient:
    """Quota-aware client. Since 2026-02-13 an API key is REQUIRED
    ($0.10/day without, $1.00/day with a free key)."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": "ORCA-GCN-CAIF/1.0 (academic research)"})
        self.s.mount("https://", HTTPAdapter(max_retries=Retry(
            total=cfg.MAX_RETRIES, backoff_factor=1.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"])))
        self.n_calls = 0
        self.last_usage = None

    def _get(self, path, params):
        params = dict(params); params["api_key"] = self.cfg.OPENALEX_API_KEY
        r = self.s.get(f"{self.cfg.OPENALEX_BASE}/{path}", params=params,
                       timeout=self.cfg.REQUEST_TIMEOUT)
        self.n_calls += 1
        for h in ("x-usage-remaining", "x-ratelimit-remaining", "x-usage-cost"):
            if h in r.headers: self.last_usage = (h, r.headers[h])
        if r.status_code == 403:
            raise RuntimeError("OpenAlex 403 — API key missing/invalid, or daily budget spent.")
        r.raise_for_status()
        return r.json()

    def paginate(self, path, params, max_results=None, desc="fetch"):
        """Cursor pagination. One call returns up to PER_PAGE results."""
        out, cursor = [], "*"
        pbar = tqdm(desc=desc, unit="rec")
        while cursor:
            p = dict(params); p.update({"per-page": self.cfg.PER_PAGE, "cursor": cursor,
                                        "select": self.cfg.SELECT_FIELDS})
            js = self._get(path, p)
            batch = js.get("results", [])
            out.extend(batch); pbar.update(len(batch))
            cursor = js.get("meta", {}).get("next_cursor")
            if not batch or (max_results and len(out) >= max_results): break
        pbar.close()
        return out[:max_results] if max_results else out

    def seed_works(self, journal_ids, y0, y1, max_papers):
        filt = (f"primary_location.source.id:{'|'.join(journal_ids)},"
                f"publication_year:{y0}-{y1}")
        return self.paginate("works", {"filter": filt},
                             max_results=max_papers, desc="seed works")

    def citing_works(self, work_ids, batch=50, max_citers=None):
        """CRITICAL: batch the reverse lookup. One call per paper would
        exceed the daily quota outright (PRD 2.3.1).

        max_citers caps the total across ALL chunks combined (None =
        unbounded, matching the original PRD design). Added after live
        testing showed a single journal's reverse-citation set can run to
        tens of thousands of records even from a small seed set -- without
        a cap this can blow the daily quota or run for a very long time."""
        ids = [w.split("/")[-1] for w in work_ids]
        out = []
        for i in tqdm(range(0, len(ids), batch), desc="citing works"):
            if max_citers is not None and len(out) >= max_citers:
                break
            chunk = "|".join(ids[i:i + batch])
            remaining = None if max_citers is None else max_citers - len(out)
            out.extend(self.paginate("works", {"filter": f"cites:{chunk}"},
                                     max_results=remaining, desc=""))
        return out[:max_citers] if max_citers is not None else out

@stage("raw_works")
def phase1_acquire(cfg):
    cl = OpenAlexClient(cfg)
    seed = cl.seed_works(cfg.JOURNAL_IDS, cfg.YEAR_START, cfg.YEAR_END,
                         cfg.MAX_SEED_PAPERS)
    log.info("seed works: %d", len(seed))
    citers = cl.citing_works([w["id"] for w in seed], cfg.CITES_BATCH, cfg.MAX_CITERS)
    log.info("citing works: %d | total API calls: %d | usage hdr: %s",
             len(citers), cl.n_calls, cl.last_usage)
    return {"seed": seed, "citers": citers,
            "seed_ids": {w["id"] for w in seed}}

RAW = phase1_acquire(CFG)


## Cell 4 — Phase 1: heterogeneous graph construction

In [ ]:
# =============================================================
# CELL 4 — PHASE 1b: typed heterogeneous graph
# =============================================================
import networkx as nx

def invert_abstract(inv_idx):
    """abstract_inverted_index -> plain text. Test 01: must tolerate None."""
    if not inv_idx: return None
    pos = {}
    for token, idxs in inv_idx.items():
        for i in idxs: pos[i] = token
    return " ".join(pos[i] for i in sorted(pos)) if pos else None

@stage("graph")
def phase1_build_graph(raw, cfg):
    G = nx.DiGraph()                     # heterogeneous, typed
    papers, authors, journals = {}, {}, {}
    all_works = {w["id"]: w for w in (raw["seed"] + raw["citers"])}

    for wid, w in all_works.items():
        loc = (w.get("primary_location") or {}).get("source") or {}
        jid = loc.get("id")
        rec = {
            "ntype": "paper",
            "year": w.get("publication_year"),
            "date": w.get("publication_date"),
            "journal": jid,
            "abstract": invert_abstract(w.get("abstract_inverted_index")),
            "title": w.get("title") or "",
            "topics": [t.get("id") for t in (w.get("topics") or []) if t.get("id")],
            "authors": [a.get("author", {}).get("id")
                        for a in (w.get("authorships") or [])],
            "institutions": [i.get("id") for a in (w.get("authorships") or [])
                             for i in (a.get("institutions") or [])],
            "cited_by_count": w.get("cited_by_count", 0),
            "is_seed": wid in raw["seed_ids"],
        }
        papers[wid] = rec
        G.add_node(wid, **rec)

        if jid and jid not in journals:
            journals[jid] = {"ntype": "journal", "name": loc.get("display_name")}
            G.add_node(jid, **journals[jid])
        if jid:
            G.add_edge(wid, jid, etype="published_in")
        for aid in rec["authors"]:
            if not aid: continue
            if aid not in authors:
                authors[aid] = {"ntype": "author"}; G.add_node(aid, **authors[aid])
            G.add_edge(wid, aid, etype="authored_by")

    # ---- citation edges: (citing) -[cites]-> (cited) ----------
    n_cite = 0
    for wid, w in all_works.items():
        for ref in (w.get("referenced_works") or []):
            if ref in all_works:                 # keep the graph closed
                G.add_edge(wid, ref, etype="cites"); n_cite += 1

    log.info("nodes: %d papers / %d authors / %d journals | cite edges: %d",
             len(papers), len(authors), len(journals), n_cite)
    return {"G": G, "papers": papers, "authors": authors, "journals": journals}

GRAPH = phase1_build_graph(RAW, CFG)

def citation_backbone(G, directed=True):
    """Paper->Paper subgraph. ORC needs the undirected projection."""
    E = [(u, v) for u, v, d in G.edges(data=True) if d.get("etype") == "cites"]
    H = nx.DiGraph() if directed else nx.Graph()
    H.add_nodes_from(n for n, d in G.nodes(data=True) if d.get("ntype") == "paper")
    H.add_edges_from(E)
    return H

DG  = citation_backbone(GRAPH["G"], directed=True)
UG  = citation_backbone(GRAPH["G"], directed=False)
log.info("backbone: %d nodes, %d directed edges, %d undirected",
         DG.number_of_nodes(), DG.number_of_edges(), UG.number_of_edges())


## Cell 5 — Phase 1: metadata extraction into four streams

In [ ]:
# =============================================================
# CELL 5 — PHASE 1c: four parallel streams
# =============================================================
from sklearn.preprocessing import MultiLabelBinarizer

@stage("streams")
def phase1_streams(graph, DG, cfg):
    P = graph["papers"]
    ids = list(DG.nodes())
    idx = {pid: i for i, pid in enumerate(ids)}

    # S_time
    s_time = pd.DataFrame([{"paper_id": p, "year": P[p]["year"],
                            "date": P[p]["date"]} for p in ids])

    # S_text  (None-tolerant: Test 01)
    s_text = {p: ((P[p]["title"] or "") + " [SEP] " + (P[p]["abstract"] or ""))
              for p in ids}
    n_missing = sum(1 for p in ids if not P[p]["abstract"])
    log.info("abstracts missing: %d / %d (%.1f%%)", n_missing, len(ids),
             100 * n_missing / max(len(ids), 1))

    # S_features : multi-hot topics
    mlb = MultiLabelBinarizer(sparse_output=False)
    X = mlb.fit_transform([P[p]["topics"] for p in ids]).astype("float32")
    if X.shape[1] == 0:                       # degenerate: no topics at all
        X = np.ones((len(ids), 1), dtype="float32")

    return {"ids": ids, "idx": idx, "s_time": s_time,
            "s_text": s_text, "X": X, "topic_vocab": list(mlb.classes_)}

STR = phase1_streams(GRAPH, DG, CFG)
log.info("feature matrix: %s", STR["X"].shape)
